# 01 — Data Diagnostics

Produces the deliverables listed in `docs/01-data-acquisition-design.md` §7:

1. **Coverage table** — per-instrument first / last observation, count, missing fraction
2. **Rolling coverage** — how many instruments are live at each date
3. **Return distribution stats** — mean, std, skew, kurtosis, ADF per instrument
4. **Full-sample correlation heatmap** — 75×75, block-ordered by bucket
5. **Rolling median pairwise correlation** — regime tell-tale

Outputs land in `memo/tables/` (CSV) and `memo/figures/` (PNG); those are the artefacts
that make it into the memo. The computation lives in `portfolio_risk.analysis.diagnostics`;
this notebook just loads the parquets, calls the helpers, and saves the results.

Pre-requisite: `python -m portfolio_risk.data.acquisition` has been run and populated
`data/processed/prices.parquet` and `data/processed/returns.parquet`.


## Setup

In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import pandas as pd

from portfolio_risk.analysis import diagnostics as diag
from portfolio_risk.data import calendar as cal
from portfolio_risk.data import paths, universe

# Figures + tables land here; both directories are tracked in git.
FIGURES_DIR = paths.PROJECT_ROOT / "memo" / "figures"
TABLES_DIR = paths.PROJECT_ROOT / "memo" / "tables"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

# Consistent style across all figures in this notebook.
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 150, "font.size": 9})

print(f"figures -> {FIGURES_DIR}")
print(f"tables  -> {TABLES_DIR}")

## Load cached data

In [ ]:
prices_path = paths.PROCESSED_DIR / "prices.parquet"
returns_path = paths.PROCESSED_DIR / "returns.parquet"

if not prices_path.exists() or not returns_path.exists():
    raise FileNotFoundError(
        f"Missing parquets under {paths.PROCESSED_DIR}. "
        "Run `python -m portfolio_risk.data.acquisition` first."
    )

prices = pd.read_parquet(prices_path)
returns = pd.read_parquet(returns_path)

print(f"prices:  {prices.shape[0]} rows x {prices.shape[1]} cols")
print(f"returns: {returns.shape[0]} rows x {returns.shape[1]} cols")
print(f"window:  {prices.index.min().date()} to {prices.index.max().date()}")

## Universe overview

In [ ]:
meta = universe.metadata_frame()
print("Bucket counts:")
print(meta["bucket"].value_counts().sort_index())

## §1 · Coverage table

For each of the 75 instruments: name, bucket, subgroup, inception (if post-2010),
first and last observed dates, number of observations, expected number of NYSE trading
days over the active period, and the missing fraction.


In [ ]:
calendar = cal.nyse_trading_days(prices.index.min(), prices.index.max())
coverage = diag.compute_coverage_table(prices, calendar)

coverage.to_csv(TABLES_DIR / "coverage.csv")
print(f"Saved coverage table with {len(coverage)} rows")

# Quick look at the instruments with the most missing data (post-2010 launches expected on top).
coverage.sort_values("pct_missing", ascending=False).head(10)

**Read**: post-2010 launches (MTUM, VIXY, SPLV, BKLN) will show the largest missing
fractions relative to the full window — expected. Any pre-2010 instrument with materially
non-zero missing warrants investigation.

## §2 · Rolling coverage

In [ ]:
rolling_coverage = diag.compute_rolling_coverage(prices)

fig = diag.plot_rolling_coverage(rolling_coverage, target=len(universe.UNIVERSE))
fig.savefig(FIGURES_DIR / "coverage_rolling.png", bbox_inches="tight")
plt.show()

print(f"Live-instrument count on first date: {rolling_coverage.iloc[0]}")
print(f"Live-instrument count on last date:  {rolling_coverage.iloc[-1]}")

## §3 · Return distribution statistics

For each instrument: daily and annualised mean and standard deviation, skew, excess kurtosis,
observed min and max, and the augmented Dickey-Fuller test statistic and p-value on the return
series. ADF is included as a sanity check — log-returns of an efficient market should be
stationary (low p-value).

In [ ]:
stats = diag.compute_return_stats(returns)
stats.to_csv(TABLES_DIR / "return_stats.csv")

# Sort by annualised volatility, high to low
stats.sort_values("std_annual", ascending=False).round(4).head(15)

In [ ]:
# ADF: how many series fail to reject the unit-root null at 5%?
n_nonstationary = (stats["adf_pvalue"] > 0.05).sum()
print(f"Instruments with ADF p > 0.05 (unit-root not rejected): {n_nonstationary} / {len(stats)}")

# Highest kurtosis (fat tails) — usually vol/spread bucket
print()
print("Top 10 by excess kurtosis:")
print(stats["excess_kurtosis"].sort_values(ascending=False).head(10).round(2))

## §4 · Full-sample correlation heatmap

Rows and columns are ordered by (bucket, subgroup) as defined in the universe module, so
bucket structure appears as blocks along the diagonal. Black lines mark bucket boundaries.

In [ ]:
full_corr = diag.compute_full_correlation(returns.dropna(how="all", axis=1))
ordered_tickers = list(full_corr.columns)
boundaries = diag.bucket_boundary_positions(ordered_tickers)

full_corr.to_csv(TABLES_DIR / "correlation_full.csv")

fig = diag.plot_correlation_heatmap(full_corr, bucket_boundaries=boundaries)
fig.savefig(FIGURES_DIR / "correlation_full.png", bbox_inches="tight")
plt.show()

**Read**: strong within-bucket correlation (bright diagonal blocks) is expected for the
sector-ETF block; equities show a mid-strength positive block; rate ETFs split into
Treasury-duration and credit sub-blocks; commodities and FX are noisier / less correlated.
Large off-block correlations flag instruments that don't behave like their bucket-mates —
worth calling out in the memo.

## §5 · Rolling median pairwise correlation

Single-number time series summarising panel-wide correlation on a 63-day rolling window.
Spikes upward correspond to "all-correlations-go-to-one" regimes (crises, coordinated
central-bank moves); troughs correspond to dispersion. This is the informal precursor to
the formal regime labelling that lands later in the project.

In [ ]:
window = diag.DEFAULT_ROLLING_CORR_WINDOW
rolling_corr = diag.compute_rolling_pairwise_correlation(
    returns.dropna(how="all", axis=1), window=window
)

fig = diag.plot_rolling_correlation(rolling_corr, window=window)
fig.savefig(FIGURES_DIR / "correlation_rolling.png", bbox_inches="tight")
plt.show()

print("Rolling median correlation summary (excl. burn-in):")
print(rolling_corr.dropna().describe().round(3))

## Summary of artefacts written

| Artefact | Path |
|---|---|
| Coverage table | `memo/tables/coverage.csv` |
| Return-distribution stats | `memo/tables/return_stats.csv` |
| Full correlation matrix | `memo/tables/correlation_full.csv` |
| Rolling coverage plot | `memo/figures/coverage_rolling.png` |
| Correlation heatmap | `memo/figures/correlation_full.png` |
| Rolling correlation plot | `memo/figures/correlation_rolling.png` |

Before committing, `Kernel → Restart Kernel and Clear All Outputs` to keep the notebook
diff-clean in git. The tracked artefacts under `memo/` are the durable outputs; the
notebook itself is only the mechanism for producing them.
